In [1]:
import os
import sys
import time

import itertools
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm as tqdm_auto
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import pytorch_lightning as pl

from IPython.display import clear_output

In [2]:
sys.path.append("../model-collection/")
from dream_wrappers import DreamRNNFacade, get_optimizer, get_scheduler

sys.path.append("../../predictor/model/")
import stability_data as stability_data
from pl_regressor import RNARegressor

In [3]:
utr_type = "stability"
seqsize = 186

## Loading data

In [4]:
PATH_FROM = f"../../predictor/regression_stability/ratios_log_stability.csv"
df = pd.read_csv(PATH_FROM)

In [5]:
Y_NAME = "log_ratio"

In [6]:
splits = dict(tuple(df.groupby('fold')))
for split_df in splits.values():
    split_df.reset_index(drop=True, inplace=True)
splits["val"].head()

,seq,fold,log_ratio
0,AAACAGGCCCCCTTCCCATCTACCTAGCCAGTACCCATCCAATGAG...,val,-0.665711
1,AAAGTCCCAAAGGCGGACCCGCTGGTGCTGGAAGCCAGAACTGTGG...,val,-0.801373
2,AAATACATATTTAGTGTTACTTGGAAAACAGCTGCTGCCAGCTAGC...,val,-0.989821
3,AACCACATGAACTGGACTGAGAGGGGGAAGAAGCGGGGAGGAAGAA...,val,-0.843702
4,AACCGGAGGGGCTGCCATTACGAGTTTACCAGCTTTTGCACGGGTA...,val,-0.737987


In [7]:
batch_size = 32
steps_per_epoch = max(1, splits["train"].shape[0] // batch_size)

epochs = None

In [8]:
num_workers = 32

In [9]:
def launch_model(
    seed: int,
    train_ds_kws: dict,
    val_ds_kws: dict,
    model_class,
    model_kws: dict,
    criterion_class,
    criterion_kws: dict,
    optimizer_class,
    optimizer_kws: dict,
    lr_scheduler_class,
    lr_scheduler_kws: dict,
    test_time_validation: bool,
    initialize_weights: bool,
    epochs: int = epochs,
):
    pl.seed_everything(seed)

    # Creating Datasets
    train_set = stability_data.StabilityData(
        df=splits["train"],
        **train_ds_kws,
    )
    val_set = stability_data.StabilityData(
        df=splits["val"],
        **val_ds_kws,
    )

    assert train_set.num_channels == val_set.num_channels
    try:
        div_factor = val_ds_kws["augment_kws"]["shift_left"] + \
                     val_ds_kws["augment_kws"]["shift_right"] + 1
    except KeyError:
        div_factor = 1

    # Creating DataLoaders
    dl_train = DataLoader(
        train_set,
        batch_size=batch_size,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True
    )
    # dl_train = utrdata.DataLoaderWrapper(dl_train, batch_per_epoch=batch_per_epoch)
    dl_val = DataLoader(
        val_set,
        batch_size=batch_size // div_factor,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False
    )

    model = RNARegressor(
        model_class=model_class,
        model_kws=model_kws | dict(
            in_channels=train_set.num_channels
        ),
        criterion_class=criterion_class,
        criterion_kws=criterion_kws,
        optimizer_class=optimizer_class,
        optimizer_kws=optimizer_kws,
        lr_scheduler_class=lr_scheduler_class,
        lr_scheduler_kws=lr_scheduler_kws,
        test_time_validation=test_time_validation,
        initialize_weights=initialize_weights,
    )
    checkpoint_callback = pl.callbacks.ModelCheckpoint(
        dirpath=f"saved_models/{utr_type}/{model.model_name}",
        save_top_k=1,
        # save_last=True,
        monitor="val_pearson_r_0",
        mode="max",
    )
    progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)

    logger = pl.loggers.tensorboard.TensorBoardLogger("tb_logs", name=model.model_name)
    trainer = pl.Trainer(
        callbacks=[checkpoint_callback, progressbar_callback],
        logger=logger,
        accelerator="gpu",
        devices=[0],
        deterministic=True,
        max_epochs=epochs,
        num_sanity_val_steps=0,
        # gradient_clip_val=1e-5,
        # gradient_clip_algorithm="norm",
    )
    trainer.fit(model=model, train_dataloaders=dl_train, val_dataloaders=dl_val)
    best_model = RNARegressor.load_from_checkpoint(checkpoint_callback.best_model_path)

    prediction = trainer.predict(model=best_model, dataloaders=dl_val)
    val_pred, val_real = zip(*prediction)
    val_pred = torch.concat(val_pred).numpy()
    val_real = torch.concat(val_real).numpy()
    val_df = splits["val"].copy()
    val_df["real_0"] = val_real[:, 0]
    val_df["pred_0"] = val_pred[:, 0]

    return trainer, val_df

In [10]:
checked = {
    "seed": [0],  # 3, 7, 13, 42, 100
    "features": [
        ("sequence",),
    ],
    "augment_dict": [
        dict(
            extend_left=0,
            extend_right=0,
            shift_left=0,
            shift_right=0,
            revcomp=False,
        ),
    ],
    "lr": [0.001],  # Decreased due to exploding gradients
    "epochs": [30],
}

for subset in itertools.product(
    *checked.values()
):
    PARAMS = dict(zip(checked.keys(), subset))
    AUGMENT_KEY = any(PARAMS["augment_dict"].values())
    AUGMENT_TEST_TIME = AUGMENT_KEY

    trainer_last, prediction_best_last = launch_model(
        seed=PARAMS["seed"],
        train_ds_kws=dict(
            features=PARAMS["features"],
            predict_cols=[Y_NAME],
        ),
        val_ds_kws=dict(
            features=PARAMS["features"],
            predict_cols=[Y_NAME],
        ),
        model_class=DreamRNNFacade,
        model_kws=dict(
            seqsize=seqsize,
            in_channels=6,
            out_channels=1,
        ),
        criterion_class=nn.MSELoss,
        criterion_kws=dict(),
        **get_optimizer(),
        **get_scheduler(
            epochs=PARAMS["epochs"],
            steps_per_epoch=steps_per_epoch,
            lr=PARAMS["lr"],
        ),
        test_time_validation=AUGMENT_TEST_TIME,
        initialize_weights=False,
        epochs=PARAMS["epochs"],
    )

Global seed set to 0
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/arsen_l/.miniconda3/envs/ml/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:612: UserWarning: Checkpoint directory /home/arsen_l/rna-legnet/wp6_clean_repo/parade-private/benchmark/stability/saved_models/stability/DreamRNNFacade exists and is not empty.
  rank_zero_warn(f"Checkpoint directory {dirpath} exists and is not empty.")
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name      | Type           | Params
-------------

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

---